## Almacenamiento de Datos Históricos en Hopsworks Feature Store (Feature Backfill)

### Por:
Dovaribi Carupia Yagari

### Fecha:
2026-08-31

### Descripción y Contexto del Requerimiento:

En una arquitectura de **Ciencia de Datos en Producción y MLOps (Patrón FTI - Feature, Training, Inference)**, el **Feature Store** actúa como la fuente centralizada de verdad para las características analíticas.

El objetivo de este notebook es realizar la carga histórica (**Backfill**) de los datos de pacientes con diagnóstico de enfermedad hepática (dataset ILPD) en el Feature Store de [Hopsworks](https://www.hopsworks.ai/).

#### Flujo del proceso implementado:
1. **Configuración segura de credenciales:** Lectura de `HOPSWORKS_API_KEY` y `HOPSWORKS_PROJECT_NAME` desde `.env` o variables de entorno.
2. **Carga y estandarización:** Lectura de los datos limpios intermedios (`data/02_intermediate/pacientes_higado_exploracion.parquet`).
3. **Preparación de features:** Estandarización de nombres en minúsculas (*snake_case*), asignación de clave primaria única (`patient_id`) y marca de tiempo (`event_time`).
4. **Definición del Feature Group:** Creación o recuperación del grupo `pacientes_higado_fg` (v2) con metadatos clínicos y formato HUDI.
5. **Inserción de datos (Backfill):** Registro de las observaciones históricas.
6. **Validación automática:** Lectura y verificación de las características almacenadas.

## 1. 📚 Importación de Librerías y Configuración de Rutas

In [ ]:
import logging
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

# Configurar el directorio raíz del proyecto en sys.path
ROOT_DIR = Path().absolute().parent.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.data.feature_store import (  # noqa: E402
    FEATURE_DESCRIPTIONS,
    get_hopsworks_project,
    get_or_create_patient_feature_group,
    prepare_patient_features_for_feature_store,
)

# Configuración de logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger("feature_backfill_notebook")

# Cargar variables de entorno desde .env en la raíz del proyecto
load_dotenv(ROOT_DIR / ".env")

## 2. 🔐 Verificación de Credenciales de Hopsworks

Para conectarse al Feature Store, se debe disponer de una cuenta gratuita en [Hopsworks Serverless](https://c.app.hopsworks.ai/) y una API Key con permisos completos (**FEATURESTORE**, **PROJECT**, **JOB**, **SERVING**).

In [ ]:
api_key = os.getenv("HOPSWORKS_API_KEY")
project_name = os.getenv("HOPSWORKS_PROJECT_NAME", "pacientes_higado_ds")

if not api_key:
    print(
        "⚠️ ADVERTENCIA: 'HOPSWORKS_API_KEY' no está configurada en las variables de entorno ni en .env."
    )
    print(
        "Por favor ingrese su API Key o agréguela al archivo .env ubicado en la raíz del proyecto:"
    )
    print("  HOPSWORKS_API_KEY=tu_api_key_aqui")
    print("  HOPSWORKS_PROJECT_NAME=pacientes_higado_ds")
else:
    print(f"✅ Credenciales detectadas para el proyecto: {project_name}")

## 3. 💾 Carga del Dataset Histórico

Cargamos los datos procesados en la etapa de exploración (`data/02_intermediate/pacientes_higado_exploracion.parquet`), o alternativamente el archivo original de `data/01_raw/` con las 11 columnas válidas.

In [ ]:
intermediate_path = ROOT_DIR / "data" / "02_intermediate" / "pacientes_higado_exploracion.parquet"
raw_path = ROOT_DIR / "data" / "01_raw" / "Pacientes_porblemas_higado_india.csv"

if intermediate_path.exists():
    print(f"Cargando datos intermedios desde: {intermediate_path}")
    df_patients = pd.read_parquet(intermediate_path)
elif raw_path.exists():
    print(f"Cargando datos raw desde: {raw_path}")
    valid_cols = [
        "Age",
        "Gender",
        "Total_Bilirubin",
        "Direct_Bilirubin",
        "Alkaline_Phosphotase",
        "Alamine_Aminotransferase",
        "Aspartate_Aminotransferase",
        "Total_Protiens",
        "Albumin",
        "Albumin_and_Globulin_Ratio",
        "Dataset",
    ]
    df_patients = pd.read_csv(raw_path, usecols=valid_cols).rename(columns={"Dataset": "Diagnosis"})
else:
    raise FileNotFoundError(
        "No se encontraron archivos de datos en data/02_intermediate/ ni data/01_raw/"
    )

print(
    f"Dimensiones del dataset cargado: {df_patients.shape[0]} filas x {df_patients.shape[1]} columnas"
)
df_patients.head()

## 4. ⚙️ Preparación de Features para el Feature Store

Hopsworks requiere:
- Nombres de columnas en minúsculas y formato *snake_case*.
- Una clave primaria (*Primary Key*) que identifique unívocamente a cada entidad (creamos `patient_id`).
- Una marca temporal de evento (*Event Time*) para consultas temporales y evitar *data leakage* en entrenamiento.
- Compatibilidad con esquemas Avro (valores nulos serializables).

In [ ]:
# Preparamos el dataframe usando la función modular de src/data/feature_store.py
# Por defecto utiliza DEFAULT_HISTORICAL_TIMESTAMP para garantizar reproducibilidad histórica
df_features = prepare_patient_features_for_feature_store(
    df_patients,
    start_id=1,
)

print("Esquema de datos preparado para Hopsworks:")
print(df_features.dtypes)
print(f"\nTotal de registros a insertar: {len(df_features)}")
df_features.head()

## 5. 🌐 Conexión a Hopsworks Feature Store

In [ ]:
# Conectar e iniciar sesión en Hopsworks
try:
    project = get_hopsworks_project(api_key=api_key, project_name=project_name)
    fs = project.get_feature_store()
    print(f"✅ Conexión establecida con el Feature Store del proyecto: '{project.name}'")
except Exception as e:
    print(f"❌ Error al conectar con Hopsworks: {e}")
    fs = None

## 6. 📦 Creación y Registro del Feature Group

Creamos u obtenemos el grupo de características `pacientes_higado_fg` versión 2 con formato HUDI para compatibilidad multiplataforma.

In [ ]:
FEATURE_GROUP_NAME = "pacientes_higado_fg"
FEATURE_GROUP_VERSION = 2

if fs is not None:
    patient_fg = get_or_create_patient_feature_group(
        fs=fs,
        name=FEATURE_GROUP_NAME,
        version=FEATURE_GROUP_VERSION,
        description="Grupo de características históricas de pacientes con variables clínicas para diagnóstico hepático",
        primary_key=["patient_id"],
        event_time="event_time",
        online_enabled=False,
        time_travel_format="HUDI",
    )
    print(f"✅ Feature Group '{FEATURE_GROUP_NAME}' v{FEATURE_GROUP_VERSION} listo para inserción.")
else:
    patient_fg = None
    print("⚠️ Omite este paso si aún no ha configurado sus credenciales de Hopsworks.")

## 7. 🚀 Inserción de Características Históricas (Backfill)

In [ ]:
if patient_fg is not None:
    print(f"Iniciando inserción de {len(df_features)} registros...")
    try:
        patient_fg.insert(df_features, write_options={"wait_for_job": True})
        print("✅ Inserción de datos históricos finalizada con éxito.")
    except Exception as err:
        print(f"❌ Error durante la inserción en Hopsworks: {err}")
        raise

    # Actualización de descripciones de features en la interfaz web de Hopsworks
    for col, desc in FEATURE_DESCRIPTIONS.items():
        if col in df_features.columns:
            try:
                patient_fg.update_feature_description(col, desc)
            except Exception as meta_err:
                print(f"⚠️ No se pudo actualizar metadato de '{col}': {meta_err}")
    print("✅ Metadatos y descripciones de características actualizados.")
else:
    print("⚠️ Feature Group no inicializado. Configure la API key para ejecutar la carga en vivo.")

## 8. 🔍 Validación y Lectura de Muestra desde el Feature Store

Validamos que los datos se encuentren registrados consultando directamente el Feature Group.

In [ ]:
if patient_fg is not None:
    print("Consultando características registradas desde Hopsworks...")
    try:
        df_read = patient_fg.read()
        print(
            f"✅ Registros recuperados con éxito: {df_read.shape[0]} filas x {df_read.shape[1]} columnas"
        )
        display(df_read.head())
    except Exception as e:
        print(f"Consulta en proceso o lectura no disponible de inmediato: {e}")
else:
    print("Muestra local de las características preparadas:")
    display(df_features.head())

## 9. 📋 Conclusiones y Siguientes Pasos

1. **Trazabilidad y Versionado:** Los datos históricos han quedado versionados en Hopsworks bajo el Feature Group `pacientes_higado_fg:2`.
2. **Preparación para Entrenamiento:** En la siguiente fase, los datos de entrenamiento se obtendrán directamente del Feature Store mediante *Feature Views*, garantizando que no exista discrepancia entre los datos de entrenamiento y los de inferencia (*training-serving skew*).
3. **Inferencia en Producción:** Las aplicaciones cliente (como `app.py` en Streamlit) o pipelines batch podrán consultar estas mismas características de manera centralizada.